# W11 Practice — MCP Servers in a LangChain Agent

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week11/W11_practice_mcp.ipynb)

**Goal.** Give an agent the tools of an MCP server. In Week 10, every tool of the agent was a Python function in the notebook. Here the notebook is an MCP host: it connects to the MCP server of the Korean Statistical Information Service (KOSIS), gets the definitions of its tools, and gives the tools to `create_agent`. Part 1 looks at the parts of MCP one at a time. Part 2 adds a tool of our own to the tools of the server.

| Part | What you learn | Main names |
|---|---|---|
| 1. The KOSIS server | the host, the client, and the server; the tool definitions that a server sends; one tool call without a model, then with an agent | `MCPAdapter`, `list_tools`, `ainvoke`, `create_agent` |
| 2. Our own tool with the server's tools | one agent with the tools of the server and a tool that saves a file | `@tool` |

**Cell labels.** The first line of each code cell says what to do with it:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.

It takes about 30 minutes and is not collected. Do the Week 10 practices first.

The server is public and needs no key. It is a pilot service, so it can be slow, refuse a call, or change its tools.

## MCP in LangChain

A tool made with `@tool` is a Python function inside one program. Another program cannot use it, so every program that needs the same data writes its own tool. The **Model Context Protocol (MCP)** is an open standard for the connection between an AI application and external systems. With MCP, a tool is written once, in a server, and every application that uses the protocol can call it.

MCP has three participants.

| Participant | Definition | In this notebook |
|---|---|---|
| MCP server | A program that provides context to MCP clients. | a public service on the internet |
| MCP client | A component that maintains a connection to one MCP server and obtains context from it for the host. | `MCPAdapter` |
| MCP host | The AI application that coordinates and manages one or more MCP clients. | this notebook, with its agent |

Tools are one kind of context that a server provides, and this notebook uses only tools.

The use of a server has three steps, and they are the same for every server.

| Step | Code |
|---|---|
| Connect to the server | `async with MCPAdapter(url) as adapter:` |
| Ask for its tools | `tools = await adapter.list_tools()` |
| Give the tools to an agent | `agent = create_agent(model, tools)` |

The tools in the list are ordinary LangChain tools. When the model calls one, the call goes to the server, and the reply of the server becomes the tool message.

A call to a server goes over the network, so these functions are asynchronous. `await` before a call waits for its reply. A notebook cell can use `await` directly.

Definitions: MCP documentation, *Architecture overview*.

## 0. Setup

*Do:* run the three cells; paste your API key into the second. The third cell makes the model object.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain[mcp]==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14"

In [ ]:
# ▶ RUN · setup: just run
import os

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

In [ ]:
# ▶ RUN · setup: just run
from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1"
model = init_chat_model(MODEL)                     # one model object for the whole notebook

## 1. The KOSIS Server

The Korean Statistical Information Service (KOSIS, 국가통계포털) of Statistics Korea (국가데이터처) holds about 1,360 kinds of approved national statistics in about 260,000 tables. Since July 2026 it has an official MCP server, as a pilot service, so that an AI application can find a statistic and get its numbers from a question in plain language. The server runs on the internet at the address below, and anyone can connect without a key.

| Participant | In this part |
|---|---|
| MCP host | this notebook |
| MCP client | the object `adapter`, made by `MCPAdapter(KOSIS_URL)` |
| MCP server | the KOSIS server at `KOSIS_URL` |

### 1.1 The tool definitions from the server

The client asks the server which tools it has, and the server answers with a definition of each tool: a name, a description for the model, and the arguments. This is the same kind of definition that `@tool` makes from a Python function, but the server wrote it.

*Do:* run the cell. It connects to the server, asks for its tools, and prints the name and the arguments of each tool, and the description of `kosis_get_data`. No tool is written in this notebook.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.mcp import MCPAdapter               # this import prints one warning: this part of LangChain is in beta

KOSIS_URL = "https://kosismcp2026.vercel.app/api/mcp"   # the address of the server, from the KOSIS guide

async with MCPAdapter(KOSIS_URL) as adapter:       # connect; the connection closes at the end of the block
    kosis_tools = await adapter.list_tools()       # ask the server for its tools; await waits for the reply

for tool in kosis_tools:
    print(tool.name, list(tool.args))              # the name of the tool, then the names of its arguments

for tool in kosis_tools:
    if tool.name == "kosis_get_data":
        print()
        print("description:", tool.description)    # the text that the model reads about this tool

### 1.2 One tool call without a model

A tool in the list is a LangChain tool object, but the code that does the work is not in the notebook. When the tool is called, the client sends its name and its arguments to the server, the server runs the tool, and the client returns the reply. The reply is a list of content blocks, and the text of a block is under the key `"text"`.

*Do:* run the cell. It calls `kosis_local_search` directly, with no model, and prints the reply of the server: the tables whose name matches the keyword.

In [ ]:
# ▶ RUN · example: read each line, then run
for tool in kosis_tools:
    if tool.name == "kosis_local_search":
        local_search = tool                        # take one tool from the list by its name

blocks = await local_search.ainvoke({"keyword": "합계출산율", "limit": 3})   # the call goes through the client to the server

print(blocks[0]["text"])                           # the first block holds the text of the reply

**Reading the output.** The reply is text that the server wrote for a model. It lists three tables with their IDs, which the next tools need. It also tells the model which tool to call next, and it asks the model to begin the answer with a notice about the pilot service. A server guides the model through these replies and through the descriptions of its tools.

### 1.3 The tools in an agent

The tools go to `create_agent` in the same way as a tool made with `@tool`. The model sees only the definitions of 1.1. When it asks for a tool, the call goes to the server as in 1.2, and the reply of the server becomes the tool message that the model reads next.

A tool result from a server can be long. The function `show` therefore prints the path of a run in short form: each tool call, the length of each tool result, and the answer.

*Do:* run the cell and read the path: which tools the model called, in which order, and with which arguments.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.agents import create_agent
from langchain.messages import HumanMessage

def show(result):
    """Print the path of one run: the tool calls, the length of each tool result, and the answer."""
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:        # one AI message can ask for several tools
                print("TOOL CALL  :", call["name"], call["args"])
        if message.type == "tool":
            print("TOOL RESULT:", message.name, "-", len(message.text), "characters")
    print()
    print(result["messages"][-1].text)             # the last message is the answer

kosis_agent = create_agent(model, kosis_tools)     # the tools of the server, like any other tools

QUESTION = "2015년부터 2024년까지 우리나라 합계출산율은 어떻게 변했나?"
result = await kosis_agent.ainvoke({"messages": [HumanMessage(QUESTION)]})   # the tools are asynchronous, so the agent runs with ainvoke
show(result)

**Reading the output.** In our run the model called four tools in the order that the replies of the server suggested: `kosis_local_search` found the table, `kosis_search` and `kosis_validate` checked it, and `kosis_get_data` returned the numbers. The answer begins with the notice of the pilot service and ends with the source and a link, as the server asked. The numbers come from the KOSIS table, not from what the model learned in training.

## 2. Our Own Tool with the Server's Tools

The KOSIS server finds statistics and returns their numbers, but it cannot write a file in this notebook. A tool from a server and a tool made with `@tool` are the same kind of object, so one agent can have both.

### 2.1 A tool of our own

*Do:* run the cell. `save_memo` writes a text file into the folder of the notebook.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.tools import tool

@tool
def save_memo(filename: str, text: str) -> str:
    """Save a memo as a text file.

    Args:
        filename: Name of the file, for example "memo.md".
        text: The whole text of the memo.
    """
    file = open(filename, "w", encoding="utf-8")   # "w": make the file, or replace an old one
    file.write(text)
    file.close()
    return "saved " + filename

print(save_memo.invoke({"filename": "test.md", "text": "a test"}))

### 2.2 One agent with both kinds of tools

*Do:* read `MEMO_QUESTION`, then run the cell. Read the path: which steps use the tools of the server, and which use our own tool. The last lines print the file that the agent saved.

In [ ]:
# ▶ RUN · example: read each line, then run
agent_tools = kosis_tools + [save_memo]            # the tools of the server and our tool in one list
memo_agent = create_agent(model, agent_tools)

MEMO_QUESTION = "2015년부터 2024년까지 출생아 수와 합계출산율을 연도별 표로 만들어 births.md 파일로 저장해줘."
result = await memo_agent.ainvoke({"messages": [HumanMessage(MEMO_QUESTION)]})
show(result)

if os.path.exists("births.md"):                    # the file that save_memo wrote
    print("----- births.md -----")
    print(open("births.md", encoding="utf-8").read())

**Reading the output.** In our run the model searched for the two statistics with the tools of the server, got the numbers of both from one KOSIS table, and then called our `save_memo` once with the whole table. The model chose among the tools of both kinds by their descriptions; it does not see which tool runs on the server and which runs in the notebook.